# EDA — Employee HR Dataset

**Secção 1: Inspeção e estrutura dos dados**

Antes de treinar os pipelines (regressão de `MonthlyIncome` e classificação de `Attrition`), é obrigatório perceber **o que entra** no modelo: quantas linhas existem, que tipos de variáveis temos, se há missing values, e quais colunas devem ser excluídas (IDs, constantes). Cada passo abaixo responde a uma pergunta concreta — igual ao que fazemos nos notebooks das aulas (Pandas, Titanic, House Prices).

In [13]:
# --- Carregamento do dataset ---
# Porquê: sem ler o CSV para um DataFrame não há EDA nem modelação.
# O path é relativo (../employee_data/) para funcionar em qualquer PC do grupo que clone o repo.
# O blind test enviará dados no mesmo formato de colunas — por isso validamos já aqui o ficheiro base.

import pandas as pd
from pathlib import Path

DATA_PATH = Path("../employee_data/employee_data.csv")
df = pd.read_csv(DATA_PATH)

print(f"Dataset carregado: {DATA_PATH}")
print(f"Registos: {len(df)} | Colunas: {len(df.columns)}")

Dataset carregado: ../employee_data/employee_data.csv
Registos: 1249 | Colunas: 35


In [14]:
# --- 1.1 head() — primeiras linhas ---
# Porquê: ver "como os dados chegam" antes de qualquer transformação.
# Serve para: confirmar nomes de colunas, ver se o CSV foi lido bem, detetar valores estranhos
# (texto onde devia ser número, categorias inesperadas) e reconhecer as variáveis-alvo
# Attrition (classificação) e MonthlyIncome (regressão).

df.head(10)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,34,No,Travel_Frequently,702,Research & Development,16,4,Life Sciences,1,838,...,3,80,0,6,3,3,5,2,1,3
1,38,No,Travel_Rarely,833,Research & Development,18,3,Medical,1,1766,...,3,80,1,15,2,3,1,0,1,0
2,51,No,Travel_Rarely,833,Research & Development,1,3,Life Sciences,1,353,...,2,80,0,1,0,2,1,0,0,0
3,60,No,Travel_Rarely,1179,Sales,16,4,Marketing,1,732,...,4,80,0,10,1,3,2,2,2,2
4,23,No,Travel_Rarely,571,Research & Development,12,2,Other,1,1982,...,3,80,0,5,6,4,5,2,1,4
5,31,No,Travel_Rarely,1003,Sales,5,3,Technical Degree,1,1749,...,3,80,1,6,3,3,5,2,0,2
6,36,No,Travel_Rarely,884,Sales,1,4,Life Sciences,1,1585,...,1,80,0,15,5,3,1,0,0,0
7,52,No,Travel_Rarely,1325,Research & Development,11,4,Life Sciences,1,813,...,2,80,1,9,3,3,5,2,1,4
8,35,No,Travel_Rarely,1017,Research & Development,6,4,Life Sciences,1,691,...,2,80,0,17,3,3,17,11,11,8
9,38,No,Non-Travel,1336,Human Resources,2,3,Human Resources,1,1805,...,4,80,3,13,3,3,11,10,3,8


In [15]:
# --- 1.2 shape — dimensão (linhas × colunas) ---
# Porquê: saber o tamanho da amostra influencia a escolha de modelos e validação cruzada.
# Poucas linhas → mais risco de overfitting; muitas colunas → mais necessidade de feature selection.
# Também documentamos no relatório quantos funcionários foram analisados.

rows, cols = df.shape
print(f"Linhas (funcionários): {rows}")
print(f"Colunas (variáveis): {cols}")
print(f"shape = {df.shape}")

Linhas (funcionários): 1249
Colunas (variáveis): 35
shape = (1249, 35)


In [16]:
# --- 1.3 columns — lista de variáveis ---
# Porquê: o pipeline sklearn recebe um DataFrame com nomes fixos; temos de saber exatamente
# quais colunas entram em X e quais são alvo (y).
# Serve para: planear ColumnTransformer (numéricas vs categóricas) e garantir que o pickle
# final aceita o mesmo schema no blind test.

print(f"Total de colunas: {len(df.columns)}\n")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:2d}. {col}")

Total de colunas: 35

 1. Age
 2. Attrition
 3. BusinessTravel
 4. DailyRate
 5. Department
 6. DistanceFromHome
 7. Education
 8. EducationField
 9. EmployeeCount
10. EmployeeNumber
11. EnvironmentSatisfaction
12. Gender
13. HourlyRate
14. JobInvolvement
15. JobLevel
16. JobRole
17. JobSatisfaction
18. MaritalStatus
19. MonthlyIncome
20. MonthlyRate
21. NumCompaniesWorked
22. Over18
23. OverTime
24. PercentSalaryHike
25. PerformanceRating
26. RelationshipSatisfaction
27. StandardHours
28. StockOptionLevel
29. TotalWorkingYears
30. TrainingTimesLastYear
31. WorkLifeBalance
32. YearsAtCompany
33. YearsInCurrentRole
34. YearsSinceLastPromotion
35. YearsWithCurrManager


In [17]:
# --- 1.4 info() — tipos e valores em falta ---
# Porquê: o pré-processamento depende do tipo de cada coluna.
#   - object → OneHotEncoder no pipeline
#   - int64/float → imputer + scaler
# A coluna "Non-Null Count" mostra missing values: se < total de linhas, precisamos de
# SimpleImputer (obrigatório no enunciado, mesmo que neste CSV esteja completo).

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1249 entries, 0 to 1248
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1249 non-null   int64 
 1   Attrition                 1249 non-null   object
 2   BusinessTravel            1249 non-null   object
 3   DailyRate                 1249 non-null   int64 
 4   Department                1249 non-null   object
 5   DistanceFromHome          1249 non-null   int64 
 6   Education                 1249 non-null   int64 
 7   EducationField            1249 non-null   object
 8   EmployeeCount             1249 non-null   int64 
 9   EmployeeNumber            1249 non-null   int64 
 10  EnvironmentSatisfaction   1249 non-null   int64 
 11  Gender                    1249 non-null   object
 12  HourlyRate                1249 non-null   int64 
 13  JobInvolvement            1249 non-null   int64 
 14  JobLevel                

In [18]:
# --- 1.5 describe() — estatísticas numéricas ---
# Porquê: variáveis com escalas muito diferentes (ex.: DailyRate vs Age) obrigam a StandardScaler.
# Quartis (25%, 50%, 75%) e max ajudam a ver outliers (ex.: salários muito altos) que podem
# distorcer a regressão de MonthlyIncome. Valores constantes (min=max, std=0) indicam colunas
# inúteis para o modelo — ex.: StandardHours sempre 80.

df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,1249.0,37.031225,9.181389,18.0,30.0,36.0,43.0,60.0
DailyRate,1249.0,803.306645,403.046710,103.0,465.0,802.0,1157.0,1499.0
DistanceFromHome,1249.0,9.274620,8.112925,1.0,2.0,7.0,14.0,29.0
Education,1249.0,2.905524,1.021740,1.0,2.0,3.0,4.0,5.0
EmployeeCount,1249.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
EmployeeNumber,1249.0,1021.179343,604.349113,1.0,486.0,1011.0,1555.0,2065.0
EnvironmentSatisfaction,1249.0,2.720576,1.092472,1.0,2.0,3.0,4.0,4.0
HourlyRate,1249.0,65.612490,20.378676,30.0,48.0,66.0,83.0,100.0
JobInvolvement,1249.0,2.735789,0.705297,1.0,2.0,3.0,3.0,4.0
JobLevel,1249.0,2.061649,1.093488,1.0,1.0,2.0,3.0,5.0


In [19]:
# --- 1.6 describe(exclude=número) — variáveis categóricas ---
# Porquê: para texto, média/desvio não faz sentido; interessa a moda (valor mais frequente)
# e quantas categorias existem (unique). Isto prepara o OneHotEncoder:
#   - "top" e "freq" mostram desbalanceamento (ex.: Attrition: maioria "No")
#   - muitas categorias em JobRole → mais colunas após encoding

import numpy as np

df.describe(exclude=[np.number]).T

,count,unique,top,freq
Attrition,1249,2,No,1048
BusinessTravel,1249,3,Travel_Rarely,879
Department,1249,3,Research & Development,814
EducationField,1249,6,Life Sciences,514
Gender,1249,2,Male,749
JobRole,1249,9,Sales Executive,282
MaritalStatus,1249,3,Married,573
Over18,1249,1,Y,1249
OverTime,1249,2,No,896


In [20]:
# --- 1.7 dtypes — tipo de cada coluna ---
# Porquê: confirma a divisão numérico vs categórico que o ColumnTransformer vai usar.
# Nota: Education e satisfações são int mas representam escalas ordinais (1–5);
# no relatório justificamos se as tratamos como numéricas ou ordinais no pipeline.

print(df.dtypes)
print("\nResumo:")
print(df.dtypes.value_counts())

Age                          int64
Attrition                   object
BusinessTravel              object
DailyRate                    int64
Department                  object
DistanceFromHome             int64
Education                    int64
EducationField              object
EmployeeCount                int64
EmployeeNumber               int64
EnvironmentSatisfaction      int64
Gender                      object
HourlyRate                   int64
JobInvolvement               int64
JobLevel                     int64
JobRole                     object
JobSatisfaction              int64
MaritalStatus               object
MonthlyIncome                int64
MonthlyRate                  int64
NumCompaniesWorked           int64
Over18                      object
OverTime                    object
PercentSalaryHike            int64
PerformanceRating            int64
RelationshipSatisfaction     int64
StandardHours                int64
StockOptionLevel             int64
TotalWorkingYears   

In [21]:
# --- 1.8 nunique() — cardinalidade ---
# Porquê: número de valores distintos decide se a coluna é útil ou deve ser removida.
#   - nunique = 1 (Over18, EmployeeCount, StandardHours) → constante, sem informação → DROP
#   - nunique ≈ nº de linhas (EmployeeNumber) → ID, não generaliza → DROP
#   - nunique = 2 (Attrition, Gender) → binária / alvo
#   - nunique alto (MonthlyRate) → quase identificador; avaliar na feature selection

cardinality = df.nunique().sort_values(ascending=False)
print(cardinality)

EmployeeNumber              1249
MonthlyRate                 1220
MonthlyIncome               1162
DailyRate                    807
HourlyRate                    71
Age                           43
TotalWorkingYears             40
YearsAtCompany                37
DistanceFromHome              29
YearsInCurrentRole            19
YearsWithCurrManager          18
YearsSinceLastPromotion       16
PercentSalaryHike             15
NumCompaniesWorked            10
JobRole                        9
TrainingTimesLastYear          7
EducationField                 6
JobLevel                       5
Education                      5
StockOptionLevel               4
JobInvolvement                 4
JobSatisfaction                4
WorkLifeBalance                4
EnvironmentSatisfaction        4
RelationshipSatisfaction       4
BusinessTravel                 3
Department                     3
MaritalStatus                  3
PerformanceRating              2
OverTime                       2
Attrition 

In [22]:
# --- 1.9 iloc — um exemplo por classe de Attrition ---
# Porquê: no notebook do IRIS comparamos uma flor de cada espécie; aqui comparamos um
# funcionário que ficou (No) vs um que saiu (Yes) lado a lado.
# Serve para: intuir quais campos parecem diferentes (OverTime, salário, anos na empresa)
# antes de gráficos e modelos — ajuda na defesa oral a explicar o problema de negócio.

idx_no = df.index[df["Attrition"] == "No"][0]
idx_yes = df.index[df["Attrition"] == "Yes"][0]

rows_by_class = df.iloc[[idx_no, idx_yes]]
rows_by_class

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,34,No,Travel_Frequently,702,Research & Development,16,4,Life Sciences,1,838,...,3,80,0,6,3,3,5,2,1,3
12,27,Yes,Travel_Rarely,1420,Sales,2,1,Marketing,1,667,...,2,80,1,5,3,3,4,3,0,2


In [23]:
# --- 1.10 sample() — amostra aleatória ---
# Porquê: head() só mostra as primeiras linhas (podem ser parecidas); sample() mostra
# linhas espalhadas pelo dataset e ajuda a detetar erros que não aparecem no topo.
# random_state=42 garante que toda a equipa vê a mesma amostra ao reexecutar a célula.

df.sample(5, random_state=42)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
679,30,No,Non-Travel,1116,Research & Development,2,3,Medical,1,571,...,3,80,0,12,2,2,11,7,6,7
1050,34,No,Travel_Rarely,131,Sales,2,3,Marketing,1,1281,...,4,80,0,4,3,3,3,2,0,2
901,57,No,Travel_Rarely,210,Sales,29,3,Marketing,1,568,...,3,80,1,32,3,2,1,0,0,0
243,34,No,Travel_Frequently,1069,Research & Development,2,1,Life Sciences,1,256,...,3,80,0,10,2,2,10,9,1,9
328,43,No,Travel_Rarely,1473,Research & Development,8,4,Other,1,526,...,4,80,0,8,3,3,5,2,0,2


In [24]:
# --- 1.11 rename() — nomes mais curtos (opcional na exploração) ---
# Porquê: no notebook de House Prices renomeiam colunas longas para ler gráficos e tabelas
# com mais facilidade. Mantemos df com nomes originais porque o blind test e o pickle
# devem receber o CSV tal como foi fornecido (mesmos nomes de colunas).
# df_renamed serve só para análises manuais neste notebook — não vai para o pipeline final.

RENAME_MAP = {
    "MonthlyIncome": "income",
    "TotalWorkingYears": "total_years",
    "YearsAtCompany": "years_company",
    "DistanceFromHome": "distance_home",
    "PercentSalaryHike": "salary_hike_pct",
    "WorkLifeBalance": "work_life_balance",
    "EnvironmentSatisfaction": "env_satisfaction",
    "JobSatisfaction": "job_satisfaction",
    "RelationshipSatisfaction": "rel_satisfaction",
}

df_renamed = df.rename(columns=RENAME_MAP)

print("Mapeamento aplicado:")
for old, new in RENAME_MAP.items():
    print(f"  {old} -> {new}")

df_renamed.filter(list(RENAME_MAP.values())).head()

Mapeamento aplicado:
  MonthlyIncome -> income
  TotalWorkingYears -> total_years
  YearsAtCompany -> years_company
  DistanceFromHome -> distance_home
  PercentSalaryHike -> salary_hike_pct
  WorkLifeBalance -> work_life_balance
  EnvironmentSatisfaction -> env_satisfaction
  JobSatisfaction -> job_satisfaction
  RelationshipSatisfaction -> rel_satisfaction


,income,total_years,years_company,distance_home,salary_hike_pct,work_life_balance,env_satisfaction,job_satisfaction,rel_satisfaction
0,2553,6,5,16,16,3,3,4,3
1,5811,15,1,18,16,3,2,4,3
2,2723,1,1,1,11,2,3,4,2
3,5405,10,2,16,14,3,1,1,4
4,2647,5,5,12,13,4,4,4,3
